# Comment trouver le bon Epsilon ($\epsilon$) pour un SVR

Dans cet exemple, nous allons utiliser la validation croisée (`GridSearchCV` de scikit-learn) pour tester systématiquement différentes valeurs de `epsilon` et trouver celle qui minimise l'erreur sur des données bruitées.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error

# 1. Génération de fausses données (Une courbe non-linéaire avec du bruit aléatoire)
np.random.seed(42)
X = np.sort(5 * np.random.rand(150, 1), axis=0)
y = np.sin(X).ravel()
# Ajout de bruit sur 1 valeur sur 5
y[::5] += 2 * (0.5 - np.random.rand(30))

# Division en jeu d'entraînement et jeu de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

plt.scatter(X, y, color='darkorange', label='Data')
plt.title('Distribution des données brutes')
plt.show()

In [ ]:
# 2. Définition de la grille de recherche (Grid Search)
# On va balayer un espace de recherche pour trouver le meilleur epsilon
param_grid = {
    'C': [1, 10, 100],               # Pénalité d'erreur
    'gamma': [0.1, 1, 'scale'],      # Coefficient du kernel RBF
    'epsilon': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0] # L'espace pour epsilon que l'on veut cibler
}

svr = SVR(kernel='rbf')

# Configuration de la validation croisée à 5 plis (5-fold CV)
grid_search = GridSearchCV(
    estimator=svr, 
    param_grid=param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
)

# Lancement de l'apprentissage compétitif
grid_search.fit(X_train, y_train)

print("Paramètres optimaux trouvés par la CV :", grid_search.best_params_)
best_epsilon = grid_search.best_params_['epsilon']

In [ ]:
# 3. Évaluation Visuelle : Bon Epsilon vs Mauvais Epsilon

# Modèle avec le paramétrage idéal trouvé
svr_optimal = SVR(kernel='rbf', C=grid_search.best_params_['C'], 
                  gamma=grid_search.best_params_['gamma'], 
                  epsilon=best_epsilon)

# Modèle avec un Epsilon volontairement trop grand (le tube tolère trop d'erreurs)
svr_bad_epsilon = SVR(kernel='rbf', C=grid_search.best_params_['C'], 
                      gamma=grid_search.best_params_['gamma'], 
                      epsilon=1.5)

svr_optimal.fit(X_train, y_train)
svr_bad_epsilon.fit(X_train, y_train)

X_plot = np.linspace(0, 5, 200)[:, None]

plt.figure(figsize=(12, 6))
plt.scatter(X, y, color='darkorange', label='Données d\'entraînement', marker='.')

# Tracé du modèle optimal
plt.plot(X_plot, svr_optimal.predict(X_plot), color='teal', lw=2, 
         label=f'SVR Optimal (epsilon={best_epsilon})')

# Tracé du modèle sous-ajusté
plt.plot(X_plot, svr_bad_epsilon.predict(X_plot), color='red', lw=2, linestyle='--', 
         label='SVR Sous-ajusté (epsilon=1.5)')

plt.legend()
plt.title("Impact de la valeur d'Epsilon sur la qualité de la régression")
plt.show()